## Planning:
- Load data
- Maximum Parameter with Optuna
- Blending with Optuna
- Output

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.metrics import roc_auc_score

# import OS
import os

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [5]:
train = pd.read_csv(competition_path/"data/train.csv")
train_data = train[["id","is_fraud"]]

test = pd.read_csv(competition_path/"data/test.csv")
test_data = test[["id"]]

submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [6]:
folder_oof = competition_path/"oof"
folder_sub = competition_path/"sub"

In [7]:
content_oof = os.listdir(folder_oof)
content_sub = os.listdir(folder_sub)

In [19]:
for oof in content_oof:
    path_oof = folder_oof/oof
    oof_data = pd.read_csv(path_oof)
    train_data = pd.concat([train_data,oof_data.iloc[:,[1]]],axis=1)

In [21]:
for sub in content_sub:
    path_sub = folder_sub/sub
    sub_data = pd.read_csv(path_sub)
    test_data = pd.concat([test_data,sub_data.iloc[:,[1]]],axis=1)

## Data Process for Modeling

In [22]:
X = train_data.drop(columns=["id","is_fraud"])
y = train_data["is_fraud"]
X_test = test_data.drop(columns=["id"])
X_test.columns = X.columns

In [23]:
# X = X.drop(columns=["log_5cv"])
# X_test = X_test.drop(columns=["log_5cv"])

In [24]:
oof_matrix = X.values 
y_true_array = y.values
num_models = oof_matrix.shape[1]

In [25]:
def objective_optuna(trial):
  weights = [trial.suggest_float(f"w_{i}", 0.0, 1.0) for i in range(num_models)]
  sum_w = sum(weights)
  if sum_w == 0:
    norm_w = weights
  else:
    norm_w = [w / sum_w for w in weights]

  blend = np.dot(X.values, norm_w)
  return roc_auc_score(y, blend)  # Optuna maximizes by default

In [26]:
study = optuna.create_study(direction="maximize")
study.optimize(objective_optuna, n_trials=1500)

[I 2026-11-10 17:33:27,693] A new study created in memory with name: no-name-d02c634c-27cd-4564-bc6f-59d220a1369f
[I 2026-11-10 17:33:28,177] Trial 0 finished with value: 0.9312583520215281 and parameters: {'w_0': 0.5050942485944159, 'w_1': 0.5360495546306723, 'w_2': 0.2595090882034172, 'w_3': 0.15904058766077867, 'w_4': 0.502635681183373, 'w_5': 0.4127221912523086, 'w_6': 0.8359319932899746, 'w_7': 0.9607856070548783, 'w_8': 0.05465706695268013, 'w_9': 0.26881018285350566, 'w_10': 0.23865648215897717, 'w_11': 0.5885134705805268, 'w_12': 0.08589945668436849, 'w_13': 0.692067012695777, 'w_14': 0.7862006533536597, 'w_15': 0.34768325514777665, 'w_16': 0.15037867412142514, 'w_17': 0.5157877390025882, 'w_18': 0.4041091784047405, 'w_19': 0.2524781116556539, 'w_20': 0.3777642099903754, 'w_21': 0.6705942135038048, 'w_22': 0.7623849334891526, 'w_23': 0.4367113432680827, 'w_24': 0.6029013996808548, 'w_25': 0.2821821258611874, 'w_26': 0.22247797435843286, 'w_27': 0.5078317846724267, 'w_28': 0.632

In [27]:
# 1. Get the best trial from your study
best_trial = study.best_trial

# 2. Extract the weights corresponding to the best trial's parameters
best_params = best_trial.params
raw_best_weights = np.array([best_params[f"w_{i}"] for i in range(num_models)])


In [28]:
# 3. Normalize the weights so they sum to 1 (matching your objective function logic)
sum_w = np.sum(raw_best_weights)
if sum_w == 0:
  final_weights = raw_best_weights
else:
  final_weights = raw_best_weights / sum_w


In [29]:
# 4. Print the best validation score and weights
print(f"Best Optuna ROC-AUC Score: {best_trial.value:.4f}")
print(f"Optimized Weights: {np.round(final_weights, 4)}")


Best Optuna ROC-AUC Score: 0.9466
Optimized Weights: [0.0012 0.0185 0.0016 0.0004 0.0097 0.0139 0.0249 0.0019 0.0235 0.0234
 0.022  0.0198 0.0008 0.0005 0.0063 0.0194 0.0113 0.0048 0.002  0.0014
 0.0068 0.0005 0.0108 0.0223 0.0263 0.016  0.0192 0.0247 0.0157 0.0119
 0.0244 0.0216 0.0178 0.0267 0.0265 0.0257 0.0007 0.0234 0.0047 0.0172
 0.0009 0.0104 0.0087 0.0078 0.0229 0.0128 0.009  0.0148 0.0263 0.0083
 0.0072 0.0012 0.0022 0.0216 0.0031 0.0025 0.0017 0.0214 0.0084 0.0141
 0.0133 0.025  0.0242 0.0223 0.0211 0.0143 0.0211 0.0177 0.0201 0.0041
 0.0015 0.0227 0.0031 0.     0.0158 0.0183]


In [30]:
# 5. Compute the final blended predictions on the test set
# X_test contains your test OOF columns (excluding 'id')
final_blend_test = np.dot(X_test.values, final_weights)

# 6. Assign to your submission dataframe and save
submit["is_fraud"] = final_blend_test
submit.to_csv(competition_path / "Blending Optuna v5.csv", index=False)
print("Submission file successfully saved!")

Submission file successfully saved!


In [ ]:
# v3: 0.9470
# v4: 0.9469
# v5: 0.9466